# 01 — Retail Data Preparation
Chạy các cell theo thứ tự trên Google Colab. Khi được yêu cầu, tải lên `retail_analytical_dataset.csv`.

**Mục tiêu:** kiểm tra dữ liệu ở cấp dòng sản phẩm trong đơn hàng, chuyển kiểu ngày, giữ nguyên dữ liệu thiếu theo kênh bán và tạo file dùng cho phân tích tiếp theo.

File gốc được giữ trong `raw`; dữ liệu xử lý nằm trong `df`. Notebook không tự xóa dòng hoặc điền giá trị thiếu. Dữ liệu năm 2021 kết thúc ngày 20/02 nên không so sánh trực tiếp với cả năm trước.


In [ ]:
import io
import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option('display.max_columns', 40)
pd.set_option('display.float_format', lambda x: f'{x:,.4f}')


## 1. Tải CSV lên
Chọn đúng một file CSV đã export từ phần A của Query 2. Colab sẽ đọc dòng đầu làm header; `NULL` và ô trống được đọc thành giá trị thiếu.


In [ ]:
from google.colab import files

uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError('Hãy tải lên đúng một file CSV.')
filename = next(iter(uploaded))
raw = pd.read_csv(io.BytesIO(uploaded[filename]), encoding='utf-8-sig', na_values=['NULL'])
print(f'File: {filename} | Shape: {raw.shape}')
display(raw.head())


## 2. Kiểm tra cấu trúc và kiểu dữ liệu

In [ ]:
expected_columns = [
    'order_number', 'line_item', 'order_date', 'order_year', 'order_month',
    'order_month_start', 'month_name', 'day_of_week', 'sales_channel',
    'delivery_date', 'delivery_days', 'customer_key', 'gender',
    'customer_city', 'customer_state', 'customer_country',
    'customer_continent', 'birthday', 'store_key', 'store_country',
    'store_state', 'square_meters', 'open_date', 'product_key',
    'product_name', 'brand', 'color', 'subcategory', 'category',
    'quantity', 'unit_price_usd', 'unit_cost_usd', 'revenue',
    'total_cost', 'gross_profit', 'line_gross_margin'
]
assert raw.columns.tolist() == expected_columns, 'Header khác cấu trúc dự kiến; kiểm tra lại file.'
assert len(raw) > 0, 'File không có dòng dữ liệu.'
print('Số dòng dự kiến của bản export hiện tại: 62,884')
print('Số dòng thực tế:', len(raw))
display(pd.DataFrame({'dtype': raw.dtypes.astype(str), 'missing': raw.isna().sum()}))


## 3. Chuyển kiểu ngày và số
Nếu gặp giá trị không chuyển được, notebook dừng để bạn kiểm tra thay vì âm thầm biến thành giá trị thiếu. Các khóa định danh dùng số nguyên có hỗ trợ giá trị thiếu; không dùng chúng như đại lượng để tính trung bình.


In [ ]:
df = raw.copy(deep=True)
date_columns = ['order_date', 'order_month_start', 'delivery_date', 'birthday', 'open_date']
for col in date_columns:
    df[col] = pd.to_datetime(df[col], format='%Y-%m-%d', errors='raise')

integer_columns = ['order_number', 'line_item', 'order_year', 'order_month',
                   'customer_key', 'store_key', 'product_key', 'quantity', 'delivery_days']
for col in integer_columns:
    df[col] = pd.to_numeric(df[col], errors='raise').astype('Int64')

numeric_columns = ['square_meters', 'unit_price_usd', 'unit_cost_usd',
                   'revenue', 'total_cost', 'gross_profit', 'line_gross_margin']
for col in numeric_columns:
    df[col] = pd.to_numeric(df[col], errors='raise')

display(df.dtypes.astype(str).rename('dtype').to_frame())
print('Khoảng ngày đặt hàng:', df.order_date.min(), 'đến', df.order_date.max())


## 4. Kiểm tra dữ liệu thiếu và dòng trùng
Trong bản CSV đã kiểm tra, cửa hàng vật lý thiếu thông tin giao hàng; kênh Online thiếu diện tích cửa hàng. Giữ các giá trị này là missing, không gán thành 0 và không dùng `dropna()` cho toàn bộ bảng.


In [ ]:
missing = df.isna().sum().rename('missing_count').to_frame()
missing['missing_pct'] = missing['missing_count'] / len(df) * 100
display(missing.query('missing_count > 0'))

channel_missing = df.groupby('sales_channel', dropna=False).agg(
    rows=('order_number', 'size'),
    missing_delivery_date=('delivery_date', lambda x: x.isna().sum()),
    missing_delivery_days=('delivery_days', lambda x: x.isna().sum()),
    missing_square_meters=('square_meters', lambda x: x.isna().sum())
)
display(channel_missing)

print('Dòng trùng hoàn toàn:', df.duplicated().sum())
print('Khóa order_number + line_item bị trùng:', df.duplicated(['order_number', 'line_item']).sum())


## 5. Kiểm tra quy tắc và công thức
Các kiểm tra dưới đây xác nhận tính nhất quán **bên trong CSV**. Chúng chưa thay thế việc đối chiếu với kết quả SQL độc lập. Nếu có lỗi, notebook hiển thị dòng liên quan và dừng trước bước xuất file.


In [ ]:
critical = ['order_number', 'line_item', 'order_date', 'order_month_start',
            'order_year', 'order_month', 'customer_key', 'product_key', 'store_key',
            'quantity', 'unit_price_usd', 'unit_cost_usd', 'revenue', 'total_cost',
            'gross_profit', 'line_gross_margin', 'sales_channel']
checks = {
    'Missing ở cột thiết yếu': df[critical].isna().any(axis=1),
    'Khóa dòng trùng': df.duplicated(['order_number', 'line_item'], keep=False),
    'Số lượng không dương': df.quantity.le(0),
    'Đơn giá hoặc giá vốn âm': df.unit_price_usd.lt(0) | df.unit_cost_usd.lt(0),
    'Năm/tháng không khớp ngày': df.order_year.ne(df.order_date.dt.year) | df.order_month.ne(df.order_date.dt.month),
    'Ngày đầu tháng không khớp': df.order_month_start.ne(df.order_date.dt.to_period('M').dt.to_timestamp()),
    'Kênh bán không khớp store_key': df.sales_channel.ne(pd.Series(np.where(df.store_key.eq(0).fillna(False), 'Online', 'Physical Store'), index=df.index)),
    'Ngày giao trước ngày đặt': df.delivery_date.lt(df.order_date),
    'Ngày giao và số ngày giao thiếu không đồng bộ': df.delivery_date.isna().ne(df.delivery_days.isna()),
    'Số ngày giao không khớp': df.delivery_date.notna() & df.delivery_days.ne((df.delivery_date - df.order_date).dt.days),
}
for label, actual, expected in [
    ('Doanh thu không khớp', df.revenue, df.quantity * df.unit_price_usd),
    ('Giá vốn không khớp', df.total_cost, df.quantity * df.unit_cost_usd),
    ('Lợi nhuận không khớp', df.gross_profit, df.revenue - df.total_cost),
]:
    checks[label] = pd.Series(~np.isclose(actual.to_numpy(dtype=float, na_value=np.nan),
                                         expected.to_numpy(dtype=float, na_value=np.nan),
                                         atol=0.01, rtol=0), index=df.index)

margin_expected = df.gross_profit / df.revenue.replace(0, np.nan)
checks['Biên lợi nhuận không khớp'] = pd.Series(
    ~np.isclose(df.line_gross_margin, margin_expected, atol=1e-8, rtol=0, equal_nan=True), index=df.index)
check_summary = pd.Series({name: int(mask.fillna(True).sum()) for name, mask in checks.items()}, name='failed_rows')
display(check_summary.to_frame())
for name, mask in checks.items():
    if mask.fillna(True).any():
        print(name)
        display(df.loc[mask.fillna(True)].head(5))
assert check_summary.sum() == 0, 'Có kiểm tra chưa đạt. Xem các dòng phía trên trước khi tiếp tục.'


## 6. Tổng KPI và đối chiếu SQL
Số đơn = số `order_number` duy nhất, không phải số dòng. Gross margin tổng = tổng gross profit / tổng revenue, không phải trung bình `line_gross_margin`.

Các khoản tiền sử dụng cột USD đã export. Chạy câu SQL ở cell văn bản kế tiếp trong SSMS trên cùng nguồn dữ liệu, rồi điền kết quả vào `sql_expected`. Để `None` nếu chưa có; notebook sẽ báo **chưa đối chiếu**, không tự đánh dấu khớp.


In [ ]:
kpis = {
    'rows': len(df),
    'orders': df.order_number.nunique(),
    'customers': df.customer_key.nunique(),
    'units_sold': int(df.quantity.sum()),
    'revenue': float(df.revenue.sum()),
    'total_cost': float(df.total_cost.sum()),
    'gross_profit': float(df.gross_profit.sum()),
}
display(pd.Series(kpis, name='CSV').to_frame())
print('Gross margin tổng:', f"{kpis['gross_profit'] / kpis['revenue']:.2%}" if kpis['revenue'] else 'Không xác định')

sql_expected = {key: None for key in kpis}
# Ví dụ: sql_expected['rows'] = 62884
# Điền các giá trị thực tế từ SQL rồi chạy lại cell này.
reconciliation = pd.DataFrame({'CSV': kpis, 'SQL': sql_expected})
reconciliation['difference'] = reconciliation['CSV'] - pd.to_numeric(reconciliation['SQL'], errors='coerce')
reconciliation['status'] = [
    'Chưa đối chiếu' if sql_expected[key] is None else
    ('Khớp' if abs(reconciliation.loc[key, 'difference']) <= (0.01 if key in ['revenue', 'total_cost', 'gross_profit'] else 0) else 'Lệch')
    for key in reconciliation.index
]
display(reconciliation)


Chạy riêng câu SQL này trong SSMS, không ghép kết quả vào CSV chi tiết:

```sql
SELECT
    COUNT_BIG(*) AS rows,
    COUNT(DISTINCT s.order_number) AS orders,
    COUNT(DISTINCT s.customer_key) AS customers,
    SUM(CAST(s.quantity AS BIGINT)) AS units_sold,
    SUM(s.quantity * p.unit_price_usd) AS revenue,
    SUM(s.quantity * p.unit_cost_usd) AS total_cost,
    SUM(s.quantity * (p.unit_price_usd - p.unit_cost_usd)) AS gross_profit
FROM retails.sales s
LEFT JOIN retails.products p ON s.product_key = p.product_key
LEFT JOIN retails.customers c ON s.customer_key = c.customer_key
LEFT JOIN retails.stores st ON s.store_key = st.store_key;
```


## 7. Xuất dữ liệu đã chuẩn bị
Sắp xếp theo ngày và khóa dòng. Xuất CSV không kèm index. CSV không giữ kiểu datetime: ở notebook tiếp theo, dùng `parse_dates=date_columns` khi đọc lại. File này sẵn sàng làm đầu vào bước phân tích; hoàn tất đối chiếu SQL trước khi chốt kết quả kinh doanh.


In [ ]:
prepared = df.sort_values(['order_date', 'order_number', 'line_item']).reset_index(drop=True)
assert prepared.shape == raw.shape, 'Số dòng/cột đã thay đổi ngoài dự kiến.'
output_file = 'retail_prepared_dataset.csv'
prepared.to_csv(output_file, index=False, encoding='utf-8-sig', date_format='%Y-%m-%d')
print('Đã xuất:', output_file, '| Shape:', prepared.shape)
display(prepared.head())
files.download(output_file)
